# DLT homework — Questions 2 and 3

Run this notebook from the `Lessons/06_dlt` directory after the pipeline completes successfully. It checks the dlt table count for Question 2 and the input-token total for the Question 1 Ollama trace for Question 3.

In [1]:
import duckdb

conn = duckdb.connect("logfire_traces.duckdb")
table_count = conn.execute("""
    SELECT COUNT(*)
    FROM information_schema.tables
    WHERE table_schema = 'agent_traces'
""").fetchone()[0]

table_count

24

## Question 3 — input-token usage

Find the trace whose input was `How do I run Ollama locally?`, then sum `gen_ai.usage.input_tokens` across its LLM calls. The total must fall in the **1,500–5,000** range.

In [2]:
question = "How do I run Ollama locally?"

token_check_sql = """
    WITH target_trace AS (
        SELECT DISTINCT records.trace_id
        FROM agent_traces.records
        JOIN agent_traces.records__attributes__gen_ai_input_messages AS messages
          ON messages._dlt_parent_id = records._dlt_id
        JOIN agent_traces.records__attributes__gen_ai_input_messages__parts AS parts
          ON parts._dlt_parent_id = messages._dlt_id
        WHERE parts.content = ?
    )
    SELECT
        records.trace_id,
        SUM(COALESCE(records.attributes__gen_ai_usage_input_tokens, 0)) AS input_token_total
    FROM agent_traces.records
    JOIN target_trace USING (trace_id)
    GROUP BY records.trace_id
"""

trace_id, input_token_total = conn.execute(token_check_sql, [question]).fetchone()
expected_min, expected_max = 1_500, 5_000

result = {
    "trace_id": trace_id,
    "input_token_total": input_token_total,
    "expected_range": "1,500–5,000",
    "check_passed": expected_min <= input_token_total <= expected_max,
}
assert result["check_passed"], result
result

{'trace_id': '019f90fd592932125c43902e3424c43f',
 'input_token_total': 4140,
 'expected_range': '1,500–5,000',
 'check_passed': True}